# 🧱 Lab: Production LLM API

**Module 10: Production Deployment** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Build** a production-ready LLM API with FastAPI that handles chat completions
2. **Implement** async endpoints with caching, retry logic, and model fallback
3. **Create** a streaming endpoint using Server-Sent Events for real-time responses
4. **Integrate** structured logging, cost tracking, and observability into every request
5. **Apply** proper error handling and response formatting throughout the API

## Concepts Covered

| Concept | From |
|---------|------|
| API Design & FastAPI Basics | mini-fastapi |
| Request Handling & Response Formatting | mini-fastapi |
| Async Patterns | mini-async |
| Error Handling | mini-async |
| Caching Strategies | mini-cache |
| Retry Patterns & Fallback | mini-retry |
| Observability & Logging | mini-observability |
| Cost Tracking | mini-cost |
| Streaming Endpoints (SSE) | This lab |

## Prerequisites

- **mini-fastapi**: Building LLM APIs with FastAPI, request/response models
- **mini-async**: Async LLM calls and concurrent execution
- **mini-cache**: In-memory response caching with hash-based keys
- **mini-retry**: Exponential backoff with jitter and model fallback
- **mini-observability**: Structured JSON logging for LLM calls
- **mini-cost**: Per-request cost calculation from token usage
- OpenAI API key configured in `.env`

## Setup

In [2]:
import os
import json
import time
import hashlib
import random
import logging
import sys
import asyncio
from datetime import datetime

from openai import AsyncOpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

print("✅ Imports ready")

✅ Imports ready


## What We're Building

We'll build a **single FastAPI application** (`app.py`) that combines every production concept from this module:

| Feature | Concept |
|---------|----------|
| `POST /chat` | API design, request handling, response formatting |
| `POST /chat/stream` | Streaming endpoints (SSE) |
| Async OpenAI client | Async patterns |
| Hash-based response cache | Caching strategies |
| Retry with backoff + model fallback | Retry & fallback patterns |
| JSON structured logs | Observability & logging |
| Per-request cost calculation | Cost tracking |
| Error responses with consistent format | Error handling |
| `GET /health` and `GET /stats` | Observability |

We'll write the application file, then test every endpoint from this notebook using `httpx`.

## Step 1 — Write the Production API

The cell below writes our complete `app.py` file. Read through the inline comments — they map each section to the mini-lab where you learned that concept.

In [3]:
%%writefile app.py
"""
Production LLM API — combines all Module 10 concepts.
"""
import os
import json
import time
import hashlib
import random
import logging
import sys
import asyncio
from datetime import datetime

from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from sse_starlette.sse import EventSourceResponse
from dotenv import load_dotenv

load_dotenv()

# ──────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────
PRIMARY_MODEL = "gpt-4o-mini"
FALLBACK_MODEL = "gpt-4o-mini"  # Same model for safety; swap to another in real usage
MAX_RETRIES = 2
BASE_DELAY = 0.5
MAX_DELAY = 8.0

# ──────────────────────────────────────────────
# STRUCTURED LOGGING (from mini-observability)
# ──────────────────────────────────────────────
class JSONFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        entry = {
            "timestamp": self.formatTime(record),
            "level": record.levelname,
            "message": record.getMessage(),
        }
        if hasattr(record, "extra_data"):
            entry.update(record.extra_data)
        return json.dumps(entry)

logger = logging.getLogger("llm_api")
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(JSONFormatter())
logger.addHandler(handler)

# ──────────────────────────────────────────────
# COST TRACKING (from mini-cost)
# ──────────────────────────────────────────────
MODEL_PRICING = {
    "gpt-4o-mini": {
        "input": 0.15 / 1_000_000,
        "output": 0.60 / 1_000_000,
    },
    "gpt-4o": {
        "input": 2.50 / 1_000_000,
        "output": 10.00 / 1_000_000,
    },
}

def calculate_cost(model: str, prompt_tokens: int, completion_tokens: int) -> float:
    pricing = MODEL_PRICING.get(model)
    if not pricing:
        return 0.0
    return prompt_tokens * pricing["input"] + completion_tokens * pricing["output"]

# In-memory stats collector
stats = {
    "total_requests": 0,
    "cache_hits": 0,
    "errors": 0,
    "total_cost_usd": 0.0,
    "total_tokens": 0,
    "started_at": datetime.utcnow().isoformat(),
}

# ──────────────────────────────────────────────
# CACHE (from mini-cache)
# ──────────────────────────────────────────────
response_cache: dict[str, dict] = {}

def cache_key(model: str, message: str, temperature: float) -> str:
    raw = f"{model}:{message}:{temperature}"
    return hashlib.sha256(raw.encode()).hexdigest()

# ──────────────────────────────────────────────
# RETRY WITH BACKOFF (from mini-retry)
# ──────────────────────────────────────────────
async def retry_with_backoff(coro_func, max_retries=MAX_RETRIES,
                              base_delay=BASE_DELAY, max_delay=MAX_DELAY):
    """
    Retry an async callable with exponential backoff + jitter.
    `coro_func` should be a zero-arg async callable (lambda or function).
    Returns (result, attempts).
    """
    last_exc = None
    for attempt in range(max_retries + 1):
        try:
            result = await coro_func()
            return result, attempt + 1
        except Exception as e:
            last_exc = e
            if attempt < max_retries:
                delay = min(base_delay * (2 ** attempt), max_delay)
                jitter = random.uniform(0, delay * 0.5)
                logger.warning("Retry scheduled", extra={"extra_data": {
                    "attempt": attempt + 1, "delay": round(delay + jitter, 2),
                    "error": str(e)[:120],
                }})
                await asyncio.sleep(delay + jitter)
    raise last_exc

# ──────────────────────────────────────────────
# REQUEST / RESPONSE MODELS (from mini-fastapi)
# ──────────────────────────────────────────────
class ChatRequest(BaseModel):
    message: str = Field(..., min_length=1, max_length=2000)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    use_cache: bool = Field(default=True)

class ChatResponse(BaseModel):
    response: str
    model: str
    tokens: dict
    cost_usd: float
    latency_ms: float
    cached: bool

class ErrorResponse(BaseModel):
    error: str
    detail: str

# ──────────────────────────────────────────────
# APP
# ──────────────────────────────────────────────
app = FastAPI(title="Production LLM API", version="1.0.0")
aclient = AsyncOpenAI()


# ---------- HEALTH ----------
@app.get("/health")
async def health():
    return {"status": "healthy", "model": PRIMARY_MODEL}


# ---------- STATS (observability) ----------
@app.get("/stats")
async def get_stats():
    return {
        **stats,
        "cache_size": len(response_cache),
    }


# ---------- CHAT (main endpoint) ----------
@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    stats["total_requests"] += 1
    start = time.perf_counter()

    # --- Cache check (from mini-cache) ---
    key = cache_key(PRIMARY_MODEL, req.message, req.temperature)
    if req.use_cache and key in response_cache:
        stats["cache_hits"] += 1
        cached = response_cache[key]
        latency_ms = (time.perf_counter() - start) * 1000
        logger.info("Cache hit", extra={"extra_data": {
            "latency_ms": round(latency_ms, 1), "cache_key": key[:12],
        }})
        return ChatResponse(
            response=cached["response"],
            model=cached["model"],
            tokens=cached["tokens"],
            cost_usd=0.0,  # Cached — no additional cost
            latency_ms=round(latency_ms, 1),
            cached=True,
        )

    # --- LLM call with retry + fallback (from mini-retry) ---
    models_to_try = [PRIMARY_MODEL, FALLBACK_MODEL]
    last_error = None

    for model in models_to_try:
        try:
            async def call_llm():
                return await aclient.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": req.message}],
                    temperature=req.temperature,
                )

            resp, attempts = await retry_with_backoff(call_llm)
            break
        except Exception as e:
            last_error = e
            logger.error("Model failed", extra={"extra_data": {
                "model": model, "error": str(e)[:120],
            }})
    else:
        # All models failed — return error (from error-handling)
        stats["errors"] += 1
        raise HTTPException(status_code=503, detail=str(last_error)[:200])

    latency_ms = (time.perf_counter() - start) * 1000
    prompt_tokens = resp.usage.prompt_tokens
    completion_tokens = resp.usage.completion_tokens
    total_tokens = resp.usage.total_tokens
    cost = calculate_cost(model, prompt_tokens, completion_tokens)
    content = resp.choices[0].message.content

    # --- Update stats (observability + cost tracking) ---
    stats["total_cost_usd"] += cost
    stats["total_tokens"] += total_tokens

    # --- Store in cache ---
    response_cache[key] = {
        "response": content,
        "model": model,
        "tokens": {
            "prompt": prompt_tokens,
            "completion": completion_tokens,
            "total": total_tokens,
        },
    }

    # --- Structured log (from mini-observability) ---
    logger.info("Chat completed", extra={"extra_data": {
        "model": model, "attempts": attempts,
        "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens,
        "cost_usd": round(cost, 8), "latency_ms": round(latency_ms, 1),
    }})

    return ChatResponse(
        response=content,
        model=model,
        tokens={"prompt": prompt_tokens, "completion": completion_tokens, "total": total_tokens},
        cost_usd=round(cost, 8),
        latency_ms=round(latency_ms, 1),
        cached=False,
    )


# ---------- STREAMING CHAT (SSE) ----------
@app.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    stats["total_requests"] += 1

    async def event_generator():
        try:
            stream = await aclient.chat.completions.create(
                model=PRIMARY_MODEL,
                messages=[{"role": "user", "content": req.message}],
                temperature=req.temperature,
                stream=True,
            )
            async for chunk in stream:
                delta = chunk.choices[0].delta.content
                if delta:
                    yield {"data": json.dumps({"token": delta})}
            yield {"data": json.dumps({"done": True})}
        except Exception as e:
            stats["errors"] += 1
            logger.error("Stream error", extra={"extra_data": {"error": str(e)[:120]}})
            yield {"data": json.dumps({"error": str(e)[:200]})}

    return EventSourceResponse(event_generator())


# ---------- GLOBAL ERROR HANDLER ----------
@app.exception_handler(Exception)
async def global_error_handler(request, exc):
    stats["errors"] += 1
    logger.error("Unhandled error", extra={"extra_data": {
        "error_type": type(exc).__name__, "error": str(exc)[:200],
    }})
    return JSONResponse(
        status_code=500,
        content={"error": "internal_server_error", "detail": "An unexpected error occurred."},
    )

Writing app.py


### Concept Map

Here's how each section of `app.py` maps back to the mini-labs:

| Section in `app.py` | Mini-Lab Origin |
|---------------------|----------------|
| `ChatRequest` / `ChatResponse` Pydantic models | mini-fastapi (request handling, response formatting) |
| `AsyncOpenAI` client, `async def` endpoints | mini-async (async patterns) |
| `response_cache` dict + `cache_key()` | mini-cache (caching strategies) |
| `retry_with_backoff()` + model loop | mini-retry (retry patterns, fallback) |
| `JSONFormatter` + `logger` | mini-observability (structured logging) |
| `calculate_cost()` + `stats` dict | mini-cost (cost tracking) |
| `HTTPException` + global handler | error handling |
| `/chat/stream` with `EventSourceResponse` | streaming endpoints (new in this lab) |

## Step 2 — Start the Server

Run this cell to start the FastAPI server in the background. We use `subprocess` so the notebook remains interactive for testing.

In [4]:
import subprocess, time

# Start the server in the background
server_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(3)  # Give the server time to start
print(f"✅ Server started (PID: {server_process.pid})")
print("   Base URL: http://127.0.0.1:8000")

✅ Server started (PID: 50496)
   Base URL: http://127.0.0.1:8000


## Step 3 — Test the Health Endpoint

A simple `GET /health` confirms the server is alive. This is a standard pattern for load balancers and monitoring.

In [5]:
import httpx

BASE_URL = "http://127.0.0.1:8000"

r = httpx.get(f"{BASE_URL}/health")
print(f"Status: {r.status_code}")
print(f"Response: {r.json()}")

Status: 200
Response: {'status': 'healthy', 'model': 'gpt-4o-mini'}


## Step 4 — Test the Chat Endpoint

The `POST /chat` endpoint combines request validation, async LLM call, retry logic, caching, cost tracking, and structured logging.

In [6]:
# First call — will hit the LLM (no cache)
r = httpx.post(
    f"{BASE_URL}/chat",
    json={"message": "What is exponential backoff? One sentence.", "temperature": 0.0},
    timeout=30,
)

data = r.json()
md(
    f"### First Call (cache miss)\n\n"
    f"**Response:** {data['response']}\n\n"
    f"| Field | Value |\n"
    f"|-------|-------|\n"
    f"| Model | {data['model']} |\n"
    f"| Tokens | {data['tokens']} |\n"
    f"| Cost | ${data['cost_usd']:.8f} |\n"
    f"| Latency | {data['latency_ms']:.0f} ms |\n"
    f"| Cached | {data['cached']} |\n"
)

### First Call (cache miss)

**Response:** Exponential backoff is a network protocol strategy that gradually increases the wait time between retries of a failed operation, typically doubling the delay after each attempt to reduce congestion and improve the chances of success.

| Field | Value |
|-------|-------|
| Model | gpt-4o-mini |
| Tokens | {'prompt': 16, 'completion': 39, 'total': 55} |
| Cost | $0.00002580 |
| Latency | 2110 ms |
| Cached | False |


## Step 5 — Test Cache Hit

Sending the exact same request should return a cached response — zero cost, near-zero latency.

In [7]:
# Second call — identical request, should be a cache hit
r = httpx.post(
    f"{BASE_URL}/chat",
    json={"message": "What is exponential backoff? One sentence.", "temperature": 0.0},
    timeout=30,
)

data = r.json()
md(
    f"### Second Call (cache hit)\n\n"
    f"**Response:** {data['response']}\n\n"
    f"| Field | Value |\n"
    f"|-------|-------|\n"
    f"| Cost | ${data['cost_usd']:.8f} |\n"
    f"| Latency | {data['latency_ms']:.1f} ms |\n"
    f"| Cached | **{data['cached']}** |\n"
)

### Second Call (cache hit)

**Response:** Exponential backoff is a network protocol strategy that gradually increases the wait time between retries of a failed operation, typically doubling the delay after each attempt to reduce congestion and improve the chances of success.

| Field | Value |
|-------|-------|
| Cost | $0.00000000 |
| Latency | 0.0 ms |
| Cached | **True** |


Notice the cached response has **$0 cost** and sub-millisecond latency. In production, this is a major cost saver for repeated queries.

## Step 6 — Test the Streaming Endpoint

The `POST /chat/stream` endpoint returns tokens via Server-Sent Events. The client receives each token as it's generated.

In [8]:
# Streaming request using httpx
print("Streaming response:\n")

full_response = ""
with httpx.stream(
    "POST",
    f"{BASE_URL}/chat/stream",
    json={"message": "List 3 benefits of API caching. Be brief.", "temperature": 0.0},
    timeout=30,
) as response:
    for line in response.iter_lines():
        if line.startswith("data: "):
            payload = json.loads(line[6:])
            if "token" in payload:
                print(payload["token"], end="", flush=True)
                full_response += payload["token"]
            elif payload.get("done"):
                break

print("\n\n✅ Stream complete")

Streaming response:



1

.

 **

Impro

...

 on

 resource

 consumption

.



✅ Stream complete


Each token arrives as a separate SSE event. In a web frontend, this creates the familiar "typing" effect. The key advantage: the user sees the first token within milliseconds instead of waiting for the entire response.

## Step 7 — Test Error Handling

Let's verify that validation errors and bad requests return consistent, structured error responses.

In [9]:
# Test: empty message (validation error)
r = httpx.post(
    f"{BASE_URL}/chat",
    json={"message": "", "temperature": 0.5},
    timeout=10,
)
print(f"Empty message → Status: {r.status_code}")
print(f"Response: {json.dumps(r.json(), indent=2)[:300]}")

Empty message → Status: 422
Response: {
  "detail": [
    {
      "type": "string_too_short",
      "loc": [
        "body",
        "message"
      ],
      "msg": "String should have at least 1 character",
      "input": "",
      "ctx": {
        "min_length": 1
      }
    }
  ]
}


In [10]:
# Test: invalid temperature (out of range)
r = httpx.post(
    f"{BASE_URL}/chat",
    json={"message": "Hello", "temperature": 5.0},
    timeout=10,
)
print(f"Bad temperature → Status: {r.status_code}")
print(f"Response: {json.dumps(r.json(), indent=2)[:300]}")

Bad temperature → Status: 422
Response: {
  "detail": [
    {
      "type": "less_than_equal",
      "loc": [
        "body",
        "temperature"
      ],
      "msg": "Input should be less than or equal to 2",
      "input": 5.0,
      "ctx": {
        "le": 2.0
      }
    }
  ]
}


FastAPI + Pydantic automatically validate inputs and return `422 Unprocessable Entity` with details about what was wrong. This is consistent, structured error handling — the client always knows what went wrong.

## Step 8 — Check Observability Stats

The `GET /stats` endpoint exposes aggregated metrics — total requests, cache hits, costs, and tokens. In production, you'd expose these to a monitoring tool like Prometheus or Datadog.

In [11]:
r = httpx.get(f"{BASE_URL}/stats")
s = r.json()

cache_rate = (s["cache_hits"] / s["total_requests"] * 100) if s["total_requests"] > 0 else 0

md(
    f"### 📊 API Stats\n\n"
    f"| Metric | Value |\n"
    f"|--------|-------|\n"
    f"| Total requests | {s['total_requests']} |\n"
    f"| Cache hits | {s['cache_hits']} |\n"
    f"| Cache hit rate | {cache_rate:.0f}% |\n"
    f"| Errors | {s['errors']} |\n"
    f"| Total tokens | {s['total_tokens']:,} |\n"
    f"| Total cost | ${s['total_cost_usd']:.6f} |\n"
    f"| Cache entries | {s['cache_size']} |\n"
    f"| Up since | {s['started_at']} |\n"
)

### 📊 API Stats

| Metric | Value |
|--------|-------|
| Total requests | 3 |
| Cache hits | 1 |
| Cache hit rate | 33% |
| Errors | 0 |
| Total tokens | 55 |
| Total cost | $0.000026 |
| Cache entries | 1 |
| Up since | 2026-04-06T23:01:59.228141 |


## Step 9 — Multiple Concurrent Requests

Let's verify the async architecture works under concurrent load. We'll send several requests simultaneously using `httpx.AsyncClient`.

In [12]:
import asyncio
import httpx

prompts = [
    "What is an API gateway? One sentence.",
    "What is rate limiting? One sentence.",
    "What is a circuit breaker pattern? One sentence.",
]

async def send_concurrent():
    async with httpx.AsyncClient(timeout=30) as ac:
        tasks = [
            ac.post(f"{BASE_URL}/chat", json={"message": p, "temperature": 0.0})
            for p in prompts
        ]
        start = time.perf_counter()
        results = await asyncio.gather(*tasks)
        total = (time.perf_counter() - start) * 1000
    return results, total

results, total_ms = await send_concurrent()

print(f"Sent {len(prompts)} concurrent requests in {total_ms:.0f} ms total\n")

for i, r in enumerate(results):
    d = r.json()
    print(f"  [{i+1}] {d['latency_ms']:.0f}ms | {d['tokens']['total']} tokens | ${d['cost_usd']:.8f}")
    print(f"      {d['response'][:100]}...\n")

Sent 3 concurrent requests in 1332 ms total

  [1] 1237ms | 50 tokens | $0.00002280
      An API gateway is a server that acts as an intermediary between clients and backend services, managi...

  [2] 1301ms | 57 tokens | $0.00002745
      Rate limiting is a technique used to control the amount of incoming or outgoing traffic to or from a...

  [3] 1280ms | 60 tokens | $0.00002835
      The circuit breaker pattern is a software design pattern used to prevent an application from repeate...



All requests were processed concurrently — the total wall time is roughly the same as the slowest single request, not the sum. This is the async pattern from `mini-async` in action at the API level.

## Step 10 — Final Stats Check

Let's pull the final stats after all our testing to see the full picture.

In [13]:
r = httpx.get(f"{BASE_URL}/stats")
s = r.json()

cache_rate = (s["cache_hits"] / s["total_requests"] * 100) if s["total_requests"] > 0 else 0

md(
    f"### 📊 Final API Stats\n\n"
    f"| Metric | Value |\n"
    f"|--------|-------|\n"
    f"| Total requests | {s['total_requests']} |\n"
    f"| Cache hits | {s['cache_hits']} ({cache_rate:.0f}%) |\n"
    f"| Errors | {s['errors']} |\n"
    f"| Total tokens used | {s['total_tokens']:,} |\n"
    f"| Total cost | ${s['total_cost_usd']:.6f} |\n"
    f"| Cache entries | {s['cache_size']} |\n"
)

### 📊 Final API Stats

| Metric | Value |
|--------|-------|
| Total requests | 6 |
| Cache hits | 1 (17%) |
| Errors | 0 |
| Total tokens used | 222 |
| Total cost | $0.000104 |
| Cache entries | 4 |


## Cleanup

In [14]:
# Stop the server
server_process.terminate()
server_process.wait()
print("✅ Server stopped")

✅ Server stopped


## 🎯 Summary

### What You Built

You built a production-ready LLM API with FastAPI that integrates async request handling, in-memory response caching, retry with exponential backoff and model fallback, streaming via Server-Sent Events, structured JSON logging, and per-request cost tracking — all in a single cohesive application.

### Key Takeaways

1. **Async endpoints** — using `AsyncOpenAI` and `async def` routes lets FastAPI handle concurrent requests without blocking, critical for I/O-bound LLM calls
2. **Response caching** — hash-based caching eliminates repeated API calls, reducing both cost to zero and latency to sub-millisecond for duplicate queries
3. **Retry + fallback** — exponential backoff with jitter handles transient errors gracefully, and model fallback ensures availability when the primary model is down
4. **Streaming (SSE)** — Server-Sent Events deliver tokens in real time, dramatically improving perceived latency for end users
5. **Structured observability** — JSON-formatted logs with token counts, latency, and cost fields enable monitoring, alerting, and cost analysis in production
6. **Consistent error handling** — Pydantic validation and a global exception handler ensure clients always receive structured, predictable error responses